In [29]:
import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# =========================
# CONFIG
# =========================

FEATURES = [
    "Heart Rate Ear(BPM)",
    "GSR",
    "Object Temperature(F)"
]

WINDOW_SIZE = 30
STEP_SIZE = 15


# =========================
# LOAD
# =========================

def load_and_clean(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    def to_sec(t):
        m, s = str(t).split(":")
        return int(m) * 60 + int(s)

    df["seconds"] = df["Timestamp"].apply(to_sec)
    return df


# =========================
# FEATURE ENGINEERING (RESEARCH LEVEL)
# =========================

def sliding_window_features(df):

    data = df.copy()
    rows = []

    for start in range(
        data["seconds"].min(),
        data["seconds"].max() - WINDOW_SIZE,
        STEP_SIZE
    ):

        window = data[
            (data["seconds"] >= start) &
            (data["seconds"] < start + WINDOW_SIZE)
        ]

        if len(window) < 3:
            continue

        feat = {}

        for col in FEATURES:

            values = window[col].values
            short = col.split("(")[0].strip().replace(" ", "_")

            # ===== CORE STATISTICS =====
            feat[f"{short}_mean"] = np.mean(values)
            feat[f"{short}_std"]  = np.std(values)

            # ===== TREND (VERY IMPORTANT) =====
            x = np.arange(len(values))
            feat[f"{short}_trend"] = np.polyfit(x, values, 1)[0]

            # ===== DELTA =====
            feat[f"{short}_range"] = np.max(values) - np.min(values)

        # ===== GSR PEAK INTENSITY =====
        gsr = window["GSR"].values
        feat["GSR_peak"] = np.max(gsr) - np.mean(gsr)

        rows.append(feat)

    return pd.DataFrame(rows)


# =========================
# SUBJECT FEATURES
# =========================

def build_phys_features(folder):

    files = os.listdir(folder)

    base = next(f for f in files if "base" in f.lower())
    lunch = next(f for f in files if "lunch" in f.lower())
    stress = next(f for f in files if "stress" in f.lower())

    base_df = load_and_clean(os.path.join(folder, base))
    lunch_df = load_and_clean(os.path.join(folder, lunch))
    stress_df = load_and_clean(os.path.join(folder, stress))

    base_feat = sliding_window_features(base_df)
    lunch_feat = sliding_window_features(lunch_df)
    stress_feat = sliding_window_features(stress_df)

    if len(base_feat) == 0:
        return None

    base_mean = base_feat.mean()

    row = {}

    # ===== RELATIVE FEATURES =====
    for df, label in [
        (base_feat, "base"),
        (lunch_feat, "lunch"),
        (stress_feat, "stress")
    ]:

        for col in df.columns:

            rel = (df[col] - base_mean[col]) / (base_mean[col] + 1e-9)

            row[f"{col}_{label}_mean"] = rel.mean()
            row[f"{col}_{label}_std"]  = rel.std()

    return row


# =========================
# DATASET BUILDER
# =========================

def build_dataset(root):

    subjects = sorted([
        d for d in os.listdir(root)
        if d.startswith("C")
    ])

    rows = []

    for s in subjects:

        try:
            feat = build_phys_features(os.path.join(root, s))

            if feat is not None:
                feat["ID"] = int("".join(filter(str.isdigit, s)))
                rows.append(feat)

        except:
            continue

    return pd.DataFrame(rows)


# =========================
# MODEL (RESEARCH VERSION)
# =========================

def train(df, target):

    X = df.drop(columns=["ID"]).values
    y = target.values

    # ===== NORMALIZATION (CRITICAL) =====
    y = (y - y.mean()) / (y.std() + 1e-9)

    loo = LeaveOneOut()

    preds = []
    true = []

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        random_state=42
    )

    for tr, te in loo.split(X):

        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        preds.append(model.predict(Xte)[0])
        true.append(yte[0])

    print("\n=== RESEARCH RESULTS ===")
    print("MAE:", mean_absolute_error(true, preds))
    print("R2 :", r2_score(true, preds))


# =========================
# MAIN
# =========================

if __name__ == "__main__":

    ROOT = "."
    QUEST = "_stressed_induced score.csv"

    print("MONITOR v3 RESEARCH PIPELINE START")

    df = build_dataset(ROOT)

    target = pd.read_csv(QUEST)["How stressed have you been?"]

    train(df, target)

MONITOR v3 RESEARCH PIPELINE START

=== RESEARCH RESULTS ===
MAE: 0.7431587980746436
R2 : 0.17246953259783893


In [25]:
import pandas as pd

df = pd.read_csv("C01/C01_PF_base.csv")

print(df.columns.tolist())

print(df.head())

['Timestamp', 'Object Temperature(F)', 'Heart Rate Ear(BPM)', 'GSR']
  Timestamp  Object Temperature(F)  Heart Rate Ear(BPM)    GSR
0     00:00                  77.34                 87.0  332.0
1     00:01                  77.34                 87.0  320.0
2     00:02                  77.34                 87.0  302.0
3     00:03                  77.34                 87.0  300.0
4     00:04                  77.34                 87.0  297.0


In [26]:
import pandas as pd

df = pd.read_csv("C01/C01_PF_base.csv")

print(df.columns.tolist())

['Timestamp', 'Object Temperature(F)', 'Heart Rate Ear(BPM)', 'GSR']
